# Remove B-frames from MP4 videos (Windows)

This notebook re-encodes one or more MP4 videos **without B-frames** using `ffmpeg` (H.264 / `libx264`).

Make sure to run this script after videos have been rotated with DLC tracking of the nose.

### Requirements
- Windows with `ffmpeg` installed and available in your PATH (verify below).
- Source videos in a directory you specify (e.g., `D:\\Whisker_Asymmetry`).

### Notes
- Removing B-frames forces the encoder to use I/P frames only (`-bf 0`). File size may increase.
- Output container stays MP4; video is re-encoded with H.264; audio stream is copied.
- Outputs are saved to a subfolder `noB_mp4` alongside your inputs with filename suffix `_noB.mp4`.
- Optionally verify that the resulting file has *no* B-frames using `ffprobe`.


In [1]:
# Check that ffmpeg is available
import subprocess, shutil
for tool in ("ffmpeg", "ffprobe"):
    path = shutil.which(tool)
    print(f"{tool}: {path if path else 'NOT FOUND in PATH'}")
if not shutil.which("ffmpeg"):
    raise SystemExit(
        "ffmpeg was not found in your PATH. Install from https://ffmpeg.org/ and reopen this notebook after adding it to PATH."
    )


ffmpeg: C:\conda_envs\whisker_tracking\Library\bin\ffmpeg.EXE
ffprobe: C:\conda_envs\whisker_tracking\Library\bin\ffprobe.EXE


Define video folder path

In [2]:
# --- Configure your input folder here ---
# Example: r"D:\\Whisker_Asymmetry"  (note the raw string r'' or double backslashes)
#VIDEO_DIR = r"C:\\Users\\Mel Gonzalez\\whisker_assymetry\\WA017_ephys001"
#VIDEO_DIR = r"D:\Whisker_Asymmetry\2026_TelC_behavior\Added_April"
VIDEO_DIR = r"E:\bilat_asymmetry_analysis\raw\Sept_test_ttl\opto_testing_ttl_alignment_20_1000ms"


# If you only want to process specific files, add their base names here (leave empty for all .mp4 files)
ONLY_THESE_FILES = []  # e.g., ["video1.mp4", "trial_A.mp4"]

# Overwrite existing outputs if they already exist?
OVERWRITE = False

# Recursively search subfolders?
RECURSIVE = False

print("VIDEO_DIR:", VIDEO_DIR)
print("ONLY_THESE_FILES:", ONLY_THESE_FILES)
print("OVERWRITE:", OVERWRITE)
print("RECURSIVE:", RECURSIVE)


VIDEO_DIR: E:\bilat_asymmetry_analysis\raw\Sept_test_ttl\opto_testing_ttl_alignment_20_1000ms
ONLY_THESE_FILES: []
OVERWRITE: False
RECURSIVE: False


For Avi and Mp4 videos

In [3]:
# ...existing code...

from pathlib import Path

video_dir = Path(VIDEO_DIR)
if not video_dir.exists():
    raise FileNotFoundError(f"Input directory not found: {video_dir}")

exts = ("*.mp4", "*.avi")
candidates = []
if RECURSIVE:
    for pat in exts:
        candidates.extend(sorted(video_dir.rglob(pat)))
else:
    for pat in exts:
        candidates.extend(sorted(video_dir.glob(pat)))

# remove duplicates while preserving order
videos = list(dict.fromkeys(candidates))

if ONLY_THESE_FILES:
    only_set = {n.lower() for n in ONLY_THESE_FILES}
    videos = [p for p in videos if p.name.lower() in only_set]

print(f"Found {len(videos)} video file(s) (mp4/avi)")
for p in videos:
    print(" -", p)

# ...existing code...

Found 8 video file(s) (mp4/avi)
 - E:\bilat_asymmetry_analysis\raw\Sept_test_ttl\opto_testing_ttl_alignment_20_1000ms\MRN_opto12_M_L_20260517_1000ms_1s_int_5x0.avi
 - E:\bilat_asymmetry_analysis\raw\Sept_test_ttl\opto_testing_ttl_alignment_20_1000ms\MRN_opto12_M_L_20260517_20ms_20ms_int_5x0.avi
 - E:\bilat_asymmetry_analysis\raw\Sept_test_ttl\opto_testing_ttl_alignment_20_1000ms\MRN_opto13_M_R_20260518_1000ms_1s_int_5x_clean_wheel0.avi
 - E:\bilat_asymmetry_analysis\raw\Sept_test_ttl\opto_testing_ttl_alignment_20_1000ms\MRN_opto13_M_R_20260518_20ms_20ms_int_5x0.avi
 - E:\bilat_asymmetry_analysis\raw\Sept_test_ttl\opto_testing_ttl_alignment_20_1000ms\MRN_opto14_F_L_20260518_1000ms_1s_int_5x0.avi
 - E:\bilat_asymmetry_analysis\raw\Sept_test_ttl\opto_testing_ttl_alignment_20_1000ms\MRN_opto14_F_L_20260518_20ms_20ms_int_5x0.avi
 - E:\bilat_asymmetry_analysis\raw\Sept_test_ttl\opto_testing_ttl_alignment_20_1000ms\MRN_opto15_F_R_20260518_1000ms_1s_int_5x0.avi
 - E:\bilat_asymmetry_analysis\r

In [4]:
import subprocess

def build_ffmpeg_cmd(in_path: Path, out_path: Path):
    # Re-encode with H.264, disable B-frames, keep yuv420p for compatibility, move moov atom to the front.
    # Copy audio as-is. You can add quality controls via -crf and -preset.
    return [
        "ffmpeg",
        "-y",  # overwrite; we handle policy outside but keep -y to simplify reruns per file
        "-hide_banner",
        "-loglevel", "warning",
        "-i", str(in_path),
        "-c:v", "libx264",
        "-bf", "0",              # <-- NO B-FRAMES
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        # Optional: tune quality/speed
        "-preset", "medium",     # slower = better compression; faster = bigger files
        "-crf", "18",            # lower = higher quality (17–23 is common)
        "-c:a", "copy",          # copy audio without re-encoding
        str(out_path),
    ]

def ensure_out_path(in_path: Path) -> Path:
    out_dir = in_path.parent / "noB_mp4"
    out_dir.mkdir(exist_ok=True)
    return out_dir / f"{in_path.stem}_noB.mp4"

print("Ready to encode. Use the next cell to run.")


Ready to encode. Use the next cell to run.


Remove the b frames of the videos found in the specified folder.

In [5]:
# Run conversion for all selected videos
errors = []
for in_path in videos:
    out_path = ensure_out_path(in_path)
    if out_path.exists() and not OVERWRITE:
        print(f"[SKIP] {out_path} exists (set OVERWRITE=True to force re-encode)")
        continue
    cmd = build_ffmpeg_cmd(in_path, out_path)
    print("\n[ENCODE]", in_path.name, "→", out_path.name)
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as e:
        print("[ERROR]", in_path, e)
        errors.append((in_path, str(e)))

print("\nDone.")
if errors:
    print("Encountered errors:")
    for p, msg in errors:
        print(" -", p, "::", msg)



[ENCODE] MRN_opto12_M_L_20260517_1000ms_1s_int_5x0.avi → MRN_opto12_M_L_20260517_1000ms_1s_int_5x0_noB.mp4

[ENCODE] MRN_opto12_M_L_20260517_20ms_20ms_int_5x0.avi → MRN_opto12_M_L_20260517_20ms_20ms_int_5x0_noB.mp4

[ENCODE] MRN_opto13_M_R_20260518_1000ms_1s_int_5x_clean_wheel0.avi → MRN_opto13_M_R_20260518_1000ms_1s_int_5x_clean_wheel0_noB.mp4

[ENCODE] MRN_opto13_M_R_20260518_20ms_20ms_int_5x0.avi → MRN_opto13_M_R_20260518_20ms_20ms_int_5x0_noB.mp4

[ENCODE] MRN_opto14_F_L_20260518_1000ms_1s_int_5x0.avi → MRN_opto14_F_L_20260518_1000ms_1s_int_5x0_noB.mp4

[ENCODE] MRN_opto14_F_L_20260518_20ms_20ms_int_5x0.avi → MRN_opto14_F_L_20260518_20ms_20ms_int_5x0_noB.mp4

[ENCODE] MRN_opto15_F_R_20260518_1000ms_1s_int_5x0.avi → MRN_opto15_F_R_20260518_1000ms_1s_int_5x0_noB.mp4

[ENCODE] MRN_opto15_F_R_20260518_20ms_20ms_int_5x0.avi → MRN_opto15_F_R_20260518_20ms_20ms_int_5x0_noB.mp4

Done.


Rotate videos found in the folder by 180 degrees clockwise

In [14]:
from pathlib import Path
import shutil, subprocess

# --- settings ---
ROTATE_RECURSIVE = True     # set False to only look in VIDEO_DIR\noB_mp4
OVERWRITE_ROTATED = False   # set True to overwrite existing *_rot180.mp4 outputs

# --- sanity check ---
if not shutil.which("ffmpeg"):
    raise SystemExit("ffmpeg not found in PATH.")

base = Path(VIDEO_DIR)

if ROTATE_RECURSIVE:
    in_files = sorted(base.rglob("*_noB.mp4"))
else:
    in_files = []
    if (base / "noB_mp4").exists():
        in_files = sorted((base / "noB_mp4").glob("*_noB.mp4"))
    else:
        in_files = sorted(base.glob("*_noB.mp4"))

print(f"Found {len(in_files)} file(s) to rotate.")

errors = []
for in_path in in_files:
    out_path = in_path.with_name(f"{in_path.stem}_rot180.mp4")

    if out_path.exists() and not OVERWRITE_ROTATED:
        print(f"[SKIP] {out_path} exists (set OVERWRITE_ROTATED=True to overwrite)")
        continue

    # 180° rotation without changing resolution: hflip+vflip (no scaling)
    cmd = [
        "ffmpeg",
        "-y" if OVERWRITE_ROTATED else "-n",
        "-hide_banner",
        "-loglevel", "warning",
        "-i", str(in_path),
        "-vf", "hflip,vflip",
        "-c:v", "libx264",
        "-bf", "0",                 # keep NO B-frames
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        "-preset", "veryfast",
        "-crf", "18",
        "-c:a", "copy",
        str(out_path),
    ]

    print(f"[ROTATE] {in_path.name} → {out_path.name}")
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as e:
        print("[ERROR]", in_path, e)
        errors.append((in_path, str(e)))

print("Done.")
if errors:
    print("Encountered errors:")
    for p, msg in errors:
        print(" -", p, "::", msg)

Found 14 file(s) to rotate.
[ROTATE] WA020_R_MRN_TelC_01_pre_TelC1_noB.mp4 → WA020_R_MRN_TelC_01_pre_TelC1_noB_rot180.mp4
[ROTATE] WA020_R_MRN_TelC_04_TelC_Day101_noB.mp4 → WA020_R_MRN_TelC_04_TelC_Day101_noB_rot180.mp4
[ROTATE] WA021_L_MRN_TelC_04_TelC_Day10_notail_2_noB.mp4 → WA021_L_MRN_TelC_04_TelC_Day10_notail_2_noB_rot180.mp4
[ROTATE] WA022_L_MRN_TelC_02_TelC_Day70_noB.mp4 → WA022_L_MRN_TelC_02_TelC_Day70_noB_rot180.mp4
[ROTATE] WA022_L_MRN_TelC_03_TelC_Day80_noB.mp4 → WA022_L_MRN_TelC_03_TelC_Day80_noB_rot180.mp4
[ROTATE] WA022_L_MRN_TelC_04_TelC_Day100_noB.mp4 → WA022_L_MRN_TelC_04_TelC_Day100_noB_rot180.mp4
[ROTATE] WA023_R_MRN_TelC_02_TelC_Day70_noB.mp4 → WA023_R_MRN_TelC_02_TelC_Day70_noB_rot180.mp4
[ROTATE] WA023_R_MRN_TelC_03_TelC_Day80_noB.mp4 → WA023_R_MRN_TelC_03_TelC_Day80_noB_rot180.mp4
[ROTATE] WA023_R_MRN_TelC_04_TelC_Day100_noB.mp4 → WA023_R_MRN_TelC_04_TelC_Day100_noB_rot180.mp4
[ROTATE] WA024_L_R_MRN_TelC_02_TelC_Day70_noB.mp4 → WA024_L_R_MRN_TelC_02_TelC_Day70_n

In [8]:
# (Optional) Verify there are no B-frames in an output file using ffprobe
# Set a file to check (choose one from the noB_mp4 folder)
CHECK_FILE = None  # e.g., r"D:\\Whisker_Asymmetry\\noB_mp4\\example_noB.mp4"

import shutil, subprocess
if CHECK_FILE:
    cf = Path(CHECK_FILE)
    if not cf.exists():
        raise FileNotFoundError(cf)
    cmd = [
        "ffprobe",
        "-hide_banner", "-loglevel", "error",
        "-select_streams", "v:0",
        "-show_entries", "frame=pict_type",
        "-of", "csv",
        cf.as_posix(),
    ]
    out = subprocess.check_output(cmd).decode("utf-8", errors="ignore")
    has_b = any(
        (line.strip().endswith(",B") or line.strip().endswith("B")) for line in out.splitlines()
    )
    print("Found B-frames?", has_b)
    if not has_b:
        print("✅ No B-frames detected.")
else:
    print("Set CHECK_FILE to a produced output to verify.")


Set CHECK_FILE to a produced output to verify.


## Tips
- If you need constant frame rate, you can add `-vsync 1` and a frame rate (e.g., `-r 30`) before the output file.
- To shrink file sizes at some quality cost, raise `-crf` (e.g., 20–23).
- If your player/editor demands baseline profile, you can add `-profile:v baseline -level 3.0` (may reduce compatibility/quality).
